In [29]:
%env ECOLOGICAL_NEURO_DATA_ROOT=/media/labuser/NA_1_2025/spyglass_common

env: ECOLOGICAL_NEURO_DATA_ROOT=/media/labuser/NA_1_2025/spyglass_common


In [30]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = 'retina'

from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display
from src.ecological_neuro.interactive import epoch_selector, trial_selector
from src.ecological_neuro.plotting import add_mpl_trial_status, add_trial_status, plot_direction_maps
from src.ecological_neuro.utils import *

sns.set_theme(context="talk", style="ticks", palette="colorblind")
PALETTE = sns.color_palette("colorblind")
plt.rcParams.update({
    "figure.constrained_layout.use": True,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "axes.spines.top": False,
    "axes.spines.right": False,
})


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Load Wilbur position data

This is a local validation-fixture loader. It deliberately keeps the Spyglass-specific path and column mapping in the notebook; reusable analysis code should receive a canonical position table from an adapter instead.

In [31]:
import os

data_root_raw = os.environ.get("ECOLOGICAL_NEURO_DATA_ROOT")
if not data_root_raw:
    raise RuntimeError(
        "Set ECOLOGICAL_NEURO_DATA_ROOT to the local Spyglass data root before running this cell."
    )

data_root = Path(data_root_raw).expanduser()
position_path = data_root / "analysis" / "position" / "trialized_position_center0.csv"
excluded_columns = {"zone", "speed_norm"}

def include_position_column(column: str) -> bool:
    return column not in excluded_columns and not column.startswith("trial_progress")

if not position_path.is_file():
    raise FileNotFoundError(f"Wilbur position table not found: {position_path}")

position_df = (
    pd.read_csv(position_path, usecols=include_position_column)
    .set_index("time")
    .sort_index()
)

if not position_df.index.is_monotonic_increasing:
    raise ValueError("Position timestamps must be monotonic after sorting.")

unexpected_columns = [
    column for column in position_df.columns if not include_position_column(column)
]
if unexpected_columns:
    raise ValueError(f"Excluded columns were retained: {unexpected_columns}")

position_df.head()


,video_frame_ind,position_x,position_y,orientation,velocity_x,velocity_y,speed,linear_position,track_segment_id,projected_x_position,projected_y_position,trial_number,trial_start,trial_end,trial_duration (s),trial_label,trial_type,"trial_direction (previous, current, next)",left/right,epoch
time,,,,,,,,,,,,,,,,,,,,
1.620843e+09,83.0,130.841658,64.731791,-0.053658,6.960408,-0.472844,6.976450,326.186090,0,130.840792,65.119877,1,1.620843e+09,1.620844e+09,31.147237,error,NaN,"('middle', 'left')",NaN,2.0
1.620843e+09,84.0,130.978969,64.705679,-0.042660,6.028157,-0.518424,6.050408,326.323342,0,130.978044,65.120183,1,1.620843e+09,1.620844e+09,31.147237,error,NaN,"('middle', 'left')",NaN,2.0
1.620843e+09,85.0,131.088632,64.655361,-0.037589,4.995344,-0.563317,5.027006,326.432892,0,131.087594,65.120428,1,1.620843e+09,1.620844e+09,31.147237,error,NaN,"('middle', 'left')",NaN,2.0
1.620843e+09,86.0,131.115555,64.624931,-0.037144,3.894330,-0.606436,3.941265,326.459748,0,131.114449,65.120488,1,1.620843e+09,1.620844e+09,31.147237,error,NaN,"('middle', 'left')",NaN,2.0
1.620843e+09,87.0,131.126578,64.618861,-0.036748,2.765255,-0.646774,2.839886,326.470757,0,131.125458,65.120512,1,1.620843e+09,1.620844e+09,31.147237,error,NaN,"('middle', 'left')",NaN,2.0


## Smooth speed and derive acceleration

`acceleration` is signed tangential acceleration: the derivative of smoothed scalar speed, in cm/s². Negative values therefore indicate braking. 

Smoothing and differentiation are performed separately within each epoch/trial, so no values are carried across sleep gaps, epoch transitions, or trials. Before defining braking events, compare the prespecified sensitivity values (for example, 0.05, 0.10, and 0.15 s) without selecting the value that produces the strongest tau result.

In [32]:
SMOOTHING_SIGMA_S = 0.10  # User-changeable Gaussian sigma in seconds.
MAX_GAP_MULTIPLIER = 3.0  # Do not smooth across gaps larger than this many sample intervals.
GROUP_COLUMNS = ["epoch", "trial_number"]

if SMOOTHING_SIGMA_S < 0:
    raise ValueError("SMOOTHING_SIGMA_S must be non-negative.")

missing_group_columns = set(GROUP_COLUMNS) - set(position_df.columns)
if missing_group_columns:
    raise KeyError(f"Cannot preserve event boundaries; missing {missing_group_columns}.")

def smooth_speed_and_acceleration(group: pd.DataFrame) -> pd.DataFrame:
    """Smooth finite within-trial speed runs and differentiate them by timestamp."""
    time = group.index.to_numpy(dtype=float)
    speed = group["speed"].to_numpy(dtype=float)
    smoothed_speed = smooth_time_series(
        time, speed, smoothing_sigma_s=SMOOTHING_SIGMA_S, max_gap_multiplier=MAX_GAP_MULTIPLIER,
    )
    acceleration = differentiate_time_series(
        time, smoothed_speed, max_gap_multiplier=MAX_GAP_MULTIPLIER,
    )

    return pd.DataFrame(
        {"speed_smoothed": smoothed_speed, "acceleration": acceleration},
        index=group.index,
    )

kinematic_parts = [
    smooth_speed_and_acceleration(group)
    for _, group in position_df.groupby(GROUP_COLUMNS, sort=False, dropna=False)
]
kinematics_df = pd.concat(kinematic_parts).reindex(position_df.index)
position_df[["speed_smoothed", "acceleration"]] = kinematics_df

position_df[["speed", "speed_smoothed", "acceleration"]].describe()


,speed,speed_smoothed,acceleration
count,228758.000000,228758.000000,228758.000000
mean,20.400423,20.385356,1.548925
std,30.877516,30.472803,44.499843
min,0.000619,0.036907,-273.144679
25%,0.529886,0.612798,-4.911268
50%,2.603932,2.905458,0.097953
75%,32.688318,33.774634,7.252384
max,130.757817,121.170822,225.404209


In [ ]:
def plot_trial_kinematics(epoch, trial_number):
    trial = position_df.query("epoch == @epoch and trial_number == @trial_number")
    if trial.empty:
        return print("No matching trial.")
    status = "incorrect" if trial["trial_label"].iat[0] != "correct" else trial["trial_type"].iat[0]
    time_s = trial.index - trial.index[0]
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08)
    fig.add_scattergl(x=time_s, y=trial["speed"], mode="lines", line={"color": "#0173B2"}, name="Speed", row=1, col=1)
    fig.add_scattergl(x=time_s, y=trial["acceleration"], mode="lines", line={"color": "#D55E00"}, name="Acceleration", row=2, col=1)
    fig.update_layout(
        template="plotly_white",
        hovermode="x unified",
        width=700,
        height=450,
    )
    fig.update_yaxes(title_text="Speed (cm/s)", row=1, col=1)
    fig.update_yaxes(title_text="Acceleration (cm/s²)", row=2, col=1)
    fig.update_xaxes(title_text="Time from trial start (s)", row=2, col=1)
    add_trial_status(fig, status)
    fig.show()

kinematics_selector = trial_selector(plot_trial_kinematics, epoch=2, trial_number=3)


Output()

In [34]:
track_background = position_df.iloc[::20]

def plot_trial_acceleration_map(epoch, trial_number):
    trial = position_df.query("epoch == @epoch and trial_number == @trial_number")
    if trial.empty:
        return print("No matching trial.")
    status = "incorrect" if trial["trial_label"].iat[0] != "correct" else trial["trial_type"].iat[0]
    color_limit = trial["acceleration"].abs().quantile(0.99)
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.scatter(track_background["position_x"], track_background["position_y"], s=2, color="lightgrey")
    points = ax.scatter(trial["position_x"], trial["position_y"], s=6, c=trial["acceleration"], cmap="RdBu", vmin=-color_limit, vmax=color_limit)
    fig.colorbar(points, ax=ax, label="Acceleration (cm/s²)")
    ax.set(xlabel="Position x (cm)", ylabel="Position y (cm)", aspect="equal")
    add_mpl_trial_status(ax, status)
    plt.show()

acceleration_map_selector = trial_selector(plot_trial_acceleration_map, epoch=2, trial_number=3)


Output()

In [35]:
spatial_variable = "acceleration"  # or "speed"
max_epoch_gap_s = 1
valid_epoch_trials = [trial_id for trial_id, trial in position_df.groupby("trial_number") if trial["epoch"].nunique() == 1 and trial.index.to_series().diff().max() < max_epoch_gap_s]

def plot_epoch_direction_map(epoch):
    samples = position_df[position_df["epoch"].eq(epoch) & position_df["trial_label"].eq("correct") & position_df["trial_number"].isin(valid_epoch_trials)]
    if samples.empty:
        return print("No matching epoch.")
    units = "cm/s²" if spatial_variable == "acceleration" else "cm/s"
    plot_direction_maps(samples, track_background, variable=spatial_variable, x_column="position_x", y_column="position_y", direction_column="trial_type", directions=("outbound", "inbound"), units=units, signed=spatial_variable == "acceleration", figsize=(10, 4))
    plt.show()

epoch_direction_selector = epoch_selector(plot_epoch_direction_map, epoch=2)


Output()

## Detect reward-well braking periods

For each correct approach, the final arm is the trailing track segment. Within each epoch and terminal arm, arrival is the centroid reaching the along-arm cutoff that retains at least 90% of correct trials. Braking begins with the final sustained period of negative acceleration and ends at that cutoff. These exploratory cutoffs must be estimated from training trials only during model evaluation.

In [36]:
ENDPOINT_RETENTION_PERCENT = 90
MIN_BRAKING_DURATION_S = 0.20
MAX_ACCELERATION_INTERRUPTION_S = 0.05

correct_endpoints = position_df[position_df["trial_label"].eq("correct")].groupby(GROUP_COLUMNS, sort=False).tail(1).copy()
correct_endpoints["final_along_arm_position_cm"] = correct_endpoints["projected_x_position"]
cutoff_rows = []
for (epoch, segment_id), endpoints in correct_endpoints.groupby(["epoch", "track_segment_id"]):
    values = np.sort(endpoints["final_along_arm_position_cm"].dropna())
    cutoff_index = int(np.floor((100 - ENDPOINT_RETENTION_PERCENT) * len(values) / 100))
    cutoff = values[cutoff_index]
    retained_n = int(np.sum(values >= cutoff))
    cutoff_rows.append({"epoch": epoch, "terminal_segment_id": segment_id, "n_correct_trials": len(values),
                        "endpoint_cutoff_cm": cutoff, "retained_n": retained_n,
                        "retained_percent": 100 * retained_n / len(values)})
endpoint_cutoffs_90 = pd.DataFrame(cutoff_rows).sort_values(["epoch", "terminal_segment_id"]).reset_index(drop=True)
endpoint_cutoff_lookup = endpoint_cutoffs_90.set_index(["epoch", "terminal_segment_id"])["endpoint_cutoff_cm"]

event_rows = []
for (epoch, trial_number), trial in position_df.groupby(GROUP_COLUMNS, sort=False):
    if trial["trial_label"].iat[0] != "correct":
        continue
    terminal_segment = trial["track_segment_id"].iat[-1]
    endpoint_cutoff = endpoint_cutoff_lookup.loc[(epoch, terminal_segment)]
    final_arm = trial["track_segment_id"].eq(terminal_segment).to_numpy()
    final_arm_start = len(trial) - 1
    while final_arm_start > 0 and final_arm[final_arm_start - 1]:
        final_arm_start -= 1
    arrival_candidates = np.flatnonzero(final_arm & (np.arange(len(trial)) >= final_arm_start) & (trial["projected_x_position"].to_numpy() >= endpoint_cutoff))
    row = {"epoch": epoch, "trial_number": trial_number, "terminal_segment_id": terminal_segment, "endpoint_cutoff_cm": endpoint_cutoff}
    if arrival_candidates.size == 0:
        row["exclusion_reason"] = "endpoint_cutoff_not_reached"
        event_rows.append(row)
        continue
    event = detect_braking_period(
        trial.index.to_numpy(), trial["acceleration"].to_numpy(), final_arm, int(arrival_candidates[0]),
        min_duration_s=MIN_BRAKING_DURATION_S, max_interruption_s=MAX_ACCELERATION_INTERRUPTION_S,
    )
    if event is None:
        row["exclusion_reason"] = "no_sustained_deceleration"
    else:
        onset, arrival = trial.iloc[event.start_index], trial.iloc[event.end_index]
        row.update({"braking_onset_time": event.start_time_s, "arrival_time": event.end_time_s, "braking_duration_s": event.duration_s,
                    "onset_position_x": onset["position_x"], "onset_position_y": onset["position_y"],
                    "arrival_position_x": arrival["position_x"], "arrival_position_y": arrival["position_y"], "exclusion_reason": None})
    event_rows.append(row)

braking_events_df = pd.DataFrame(event_rows)
braking_events_df["exclusion_reason"].fillna("detected").value_counts()


exclusion_reason
detected                       185
endpoint_cutoff_not_reached     14
no_sustained_deceleration        1
Name: count, dtype: int64

In [37]:
braking_track_background = position_df.iloc[::20]

def plot_detected_braking(epoch, trial_number):
    trial = position_df.query("epoch == @epoch and trial_number == @trial_number")
    event = braking_events_df.query("epoch == @epoch and trial_number == @trial_number")
    if trial.empty or event.empty:
        return print("No matching braking event.")
    if pd.notna(event["exclusion_reason"].iat[0]):
        return print(event["exclusion_reason"].iat[0])
    color_limit = trial["acceleration"].abs().quantile(0.99)
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.scatter(braking_track_background["position_x"], braking_track_background["position_y"], s=2, color="lightgrey")
    points = ax.scatter(trial["position_x"], trial["position_y"], s=6, c=trial["acceleration"], cmap="RdBu", vmin=-color_limit, vmax=color_limit)
    ax.scatter(event["onset_position_x"], event["onset_position_y"], s=220, marker="*", color="white", edgecolor="black", linewidth=1.5, label="Braking onset", zorder=3)
    fig.colorbar(points, ax=ax, label="Acceleration (cm/s²)")
    ax.set(xlabel="Position x (cm)", ylabel="Position y (cm)", aspect="equal")
    ax.legend()
    add_mpl_trial_status(ax, trial["trial_type"].iat[0])
    plt.show()

braking_map_selector = trial_selector(plot_detected_braking, epoch=2, trial_number=6)


Output()

## Calculate metrics at braking onset

Calculate tau and its component variables at each detected onset, plus QC summaries of the complete braking period. On these straight final arms, distance to the fixed centroid endpoint is the along-path gap.

In [38]:
MIN_CLOSURE_SPEED_CM_S = 1.0

metric_rows = []
detected_events = braking_events_df[braking_events_df["exclusion_reason"].isna()]
for event in detected_events.itertuples(index=False):
    trial = position_df.query("epoch == @event.epoch and trial_number == @event.trial_number")
    time = trial.index.to_numpy()
    onset_index = int(np.searchsorted(time, event.braking_onset_time))
    arrival_index = int(np.searchsorted(time, event.arrival_time))
    remaining_gap = event.endpoint_cutoff_cm - trial["projected_x_position"]
    metrics = calculate_braking_metrics(
        time, remaining_gap, trial["speed_smoothed"], trial["acceleration"], onset_index, arrival_index,
        closure_speed_floor=MIN_CLOSURE_SPEED_CM_S, smoothing_sigma_s=SMOOTHING_SIGMA_S,
        max_gap_multiplier=MAX_GAP_MULTIPLIER,
    )
    metric_rows.append({"epoch": event.epoch, "trial_number": event.trial_number,
                        "trial_type": trial["trial_type"].iat[0], "terminal_segment_id": event.terminal_segment_id,
                        "position_x_cm": event.onset_position_x, "position_y_cm": event.onset_position_y, **vars(metrics)})

spatial_metric_names = {
    "remaining_gap_at_onset": "remaining_gap_at_onset_cm", "closure_speed_at_onset": "closure_speed_at_onset_cm_s",
    "speed_at_onset": "speed_at_onset_cm_s", "acceleration_at_onset": "acceleration_at_onset_cm_s2",
    "speed_at_arrival": "speed_at_arrival_cm_s", "speed_reduction": "speed_reduction_cm_s",
    "distance_travelled": "distance_travelled_cm", "mean_deceleration": "mean_deceleration_cm_s2",
    "max_deceleration": "max_deceleration_cm_s2",
}
braking_metrics_df = pd.DataFrame(metric_rows).rename(columns=spatial_metric_names)
braking_metrics_df.head()

breaking_onset_metrics = braking_metrics_df[[
    "epoch", "trial_number", "trial_type", "terminal_segment_id", "time_to_arrival_s",
    "remaining_gap_at_onset_cm", "closure_speed_at_onset_cm_s", "tau_at_onset_s",
    "speed_at_onset_cm_s", "acceleration_at_onset_cm_s2",
]].copy()
breaking_onset_metrics.head()


,epoch,trial_number,trial_type,terminal_segment_id,time_to_arrival_s,remaining_gap_at_onset_cm,closure_speed_at_onset_cm_s,tau_at_onset_s,speed_at_onset_cm_s,acceleration_at_onset_cm_s2
0,2.0,5,inbound,0,0.451885,27.448004,69.591563,0.394416,64.729517,-3.458236
1,2.0,6,outbound,3,0.757999,44.448270,89.081113,0.498964,86.549731,-0.301696
2,2.0,7,inbound,0,0.772577,60.020774,94.013426,0.638428,92.993798,-1.456269
3,2.0,8,outbound,4,0.670538,46.247524,94.625374,0.488743,91.631460,-2.138954
4,2.0,9,inbound,0,0.860038,54.510521,95.966379,0.568017,95.201993,-1.288895


## Braking-onset metric distributions by epoch

In [39]:
trial_info_columns = {"epoch", "trial_number", "trial_type", "terminal_segment_id"}
onset_metric_columns = [column for column in breaking_onset_metrics.columns if column not in trial_info_columns]

def plot_braking_onset_histograms(epoch):
    selected = breaking_onset_metrics[breaking_onset_metrics["epoch"].eq(epoch)]
    fig, axes = plt.subplots(2, 3, figsize=(10, 6))
    for ax, metric in zip(axes.flat, onset_metric_columns):
        sns.histplot(selected[metric].dropna(), bins=15, ax=ax, color=PALETTE[0])
        ax.set(title=metric.replace("_", " "), xlabel="", ylabel="Trials")
    for ax in axes.flat[len(onset_metric_columns):]:
        ax.remove()
    fig.suptitle(f"Braking-onset metrics — epoch {epoch} (n={len(selected)})")
    plt.show()


In [40]:

epoch_dropdown = widgets.Dropdown(options=[int(epoch) for epoch in sorted(breaking_onset_metrics["epoch"].unique())], description="Epoch")
histogram_output = widgets.interactive_output(plot_braking_onset_histograms, {"epoch": epoch_dropdown})
display(widgets.VBox([epoch_dropdown, histogram_output]))


In [41]:
def report_normalized_iqrs(epoch):
    selected = breaking_onset_metrics[breaking_onset_metrics["epoch"].eq(epoch)][onset_metric_columns]
    medians = selected.median()
    iqrs = selected.quantile(0.75) - selected.quantile(0.25)
    report = pd.DataFrame({"median": medians, "iqr": iqrs, "normalized_iqr": iqrs / medians.abs().replace(0, np.nan)})
    display(report.sort_values("normalized_iqr"))

normalized_iqr_output = widgets.interactive_output(report_normalized_iqrs, {"epoch": epoch_dropdown})
display(normalized_iqr_output)


Output()